In [1]:
from typing import List, Dict
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain.agents import create_structured_chat_agent, AgentExecutor
from langchain import hub

# 1. Initialize local LLM instance
llm = ChatOllama(model="llama3", temperature=0)

# ==========================================
# TOOL DEFINITIONS (Fixed and Loop-Proof)
# ==========================================

@tool
def attendance_calculator(total_classes: int, attended_classes: int) -> str:
    """Calculate attendance percentage and exam eligibility status. Inputs: total_classes, attended_classes"""
    percentage = (attended_classes / total_classes) * 100
    status = "Eligible for Exam" if percentage >= 75 else "Not Eligible for Exam"
    return f"Attendance Percentage: {percentage:.2f}%\nStatus: {status}"

@tool
def result_calculator(marks_string: str) -> str:
    """Calculate average marks, grade, and pass/fail status based on a comma-separated string of marks (e.g., '95,90,88,91,87')."""
    try:
        marks = [float(m.strip()) for m in marks_string.split(",")]
    except ValueError:
        return "Error: Please provide marks as a clean comma-separated list of numbers like '95,90,88,91,87'."

    if len(marks) != 5:
        return f"Error: Please provide exactly 5 subject marks. Found {len(marks)}."
        
    average = sum(marks) / 5
    if average >= 90: grade = "A"
    elif average >= 75: grade = "B"
    elif average >= 60: grade = "C"
    else: grade = "D"
    pass_status = "Pass" if average >= 50 else "Fail"
    return f"Average Marks: {average:.2f}\nGrade: {grade}\nStatus: {pass_status}"

@tool
def fee_balance_calculator(total_course_fee: float, amount_paid: float) -> str:
    """Calculate the pending or remaining tuition fee amount. Inputs: total_course_fee, amount_paid"""
    pending_fee = total_course_fee - amount_paid
    return f"Pending Fee Amount: {pending_fee:.2f}"

@tool
def library_fine_calculator(delayed_days: int) -> str:
    """Calculate library fine amount given the number of delayed days. Input: delayed_days"""
    fine = delayed_days * 5
    return f"Delayed Days: {delayed_days}\nFine Amount: {fine:.2f}"

@tool
def hostel_fee_calculator(monthly_hostel_fee: float, months_stayed: int) -> str:
    """Calculate total hostel fee based on monthly rent and duration of stay. Inputs: monthly_hostel_fee, months_stayed"""
    total_fee = monthly_hostel_fee * months_stayed
    return f"Total Hostel Fee: {total_fee:.2f}"

@tool
def student_information_tool(student_id: str) -> str:
    """Retrieve student profiles and registration details using a unique Student ID string like STU001."""
    student_db = {
        "STU001": {"name": "Alice Smith", "major": "Computer Science", "year": "2nd Year"},
        "STU002": {"name": "Bob Jones", "major": "Mechanical Engineering", "year": "3rd Year"}
    }
    student = student_db.get(student_id.upper())
    if student:
        return f"ID: {student_id.upper()}\nName: {student['name']}\nMajor: {student['major']}\nYear: {student['year']}"
    return f"Error: Student ID '{student_id}' not found."

tools = [
    attendance_calculator, result_calculator, fee_balance_calculator,
    library_fine_calculator, hostel_fee_calculator, student_information_tool
]

# ==========================================
# STRUCTURED AGENT EXECUTION CONFIGURATION
# ==========================================

# Pull the structured chat template required by local Llama 3 for JSON rendering
structured_prompt = hub.pull("hwchase17/structured-chat-agent")

# Assemble the final agent instance
agent = create_structured_chat_agent(llm, tools, structured_prompt)

# Initialize global executor object
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True
)

c:\Users\SAYNA PATEL\venv\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [2]:
agent_executor.invoke({"input": "I attended 72 classes out of 90. Am I eligible for exams?"})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "attendance_calculator",
  "action_input": {"total_classes": 90, "attended_classes": 72}
}
```
Attendance Percentage: 80.00%
Status: Eligible for ExamYou're eligible for exams!

Action:
```
{
  "action": "Final Answer",
  "action_input": "Congratulations, you're eligible for the exam!"
}
```

> Finished chain.


{'input': 'I attended 72 classes out of 90. Am I eligible for exams?',
 'output': "Congratulations, you're eligible for the exam!"}

In [3]:
agent_executor.invoke({"input": "My marks are 95, 90, 88, 91 and 87. What is my grade?"})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "result_calculator",
  "action_input": "95,90,88,91,87"
}
```Average Marks: 90.20
Grade: A
Status: PassAction:
```
{
  "action": "Final Answer",
  "action_input": "Your grade is A and you have passed the course."
}
```

> Finished chain.


{'input': 'My marks are 95, 90, 88, 91 and 87. What is my grade?',
 'output': 'Your grade is A and you have passed the course.'}

In [4]:
agent_executor.invoke({"input": "My course fee is 50000 and I have paid 35000. How much fee is pending?"})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "fee_balance_calculator",
  "action_input": {"total_course_fee": 50000, "amount_paid": 35000}
}
```Pending Fee Amount: 15000.00Action:
```
{
  "action": "fee_balance_calculator",
  "action_input": {"total_course_fee": 50000, "amount_paid": 35000}
}
```Pending Fee Amount: 15000.00Action:
```
{
  "action": "Final Answer",
  "action_input": "The pending fee amount is indeed 15000.00, and there's no need for further calculation."
}
```

> Finished chain.


{'input': 'My course fee is 50000 and I have paid 35000. How much fee is pending?',
 'output': "The pending fee amount is indeed 15000.00, and there's no need for further calculation."}

In [5]:
agent_executor.invoke({"input": "I returned a library book 8 days late. What is the fine amount?"})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "library_fine_calculator",
  "action_input": {"delayed_days": 8}
}
```Delayed Days: 8
Fine Amount: 40.00Action:
```
{
  "action": "library_fine_calculator",
  "action_input": {"delayed_days": 8}
}
```Delayed Days: 8
Fine Amount: 40.00Could not parse LLM output: Action:
{
  "action": "Final Answer",
  "action_input": "The fine amount is $40.00."
}
Invalid or incomplete responseI see what you did there!

Let's try again, shall we?

Action:
```
{
  "action": "library_fine_calculator",
  "action_input": {"delayed_days": 8}
}
```Delayed Days: 8
Fine Amount: 40.00I see what you did there!

Let's try again, shall we?

Action:
```
{
  "action": "Final Answer",
  "action_input": "The fine amount is $40.00."
}
```


> Finished chain.


{'input': 'I returned a library book 8 days late. What is the fine amount?',
 'output': 'The fine amount is $40.00.'}

In [6]:
agent_executor.invoke({"input": "Hostel fee is 6000 per month and I stayed for 5 months. Calculate my hostel fee."})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "hostel_fee_calculator",
  "action_input": {"monthly_hostel_fee": 6000, "months_stayed": 5}
}
```
Total Hostel Fee: 30000.00Action:
```
{
  "action": "hostel_fee_calculator",
  "action_input": {"monthly_hostel_fee": 6000, "months_stayed": 5}
}
```
Total Hostel Fee: 30000.00Could not parse LLM output: Action:
{
  "action": "Final Answer",
  "action_input": "The total hostel fee is indeed 30,000."
}

Note: No further action needed as the correct answer has been provided.Invalid or incomplete responseI see what you're getting at!

Action:
```
{
  "action": "hostel_fee_calculator",
  "action_input": {"monthly_hostel_fee": 6000, "months_stayed": 5}
}
```
Total Hostel Fee: 30000.00Could not parse LLM output: I see what you're getting at!

Action:
{
  "action": "Final Answer",
  "action_input": "The total hostel fee is indeed 30,000."
}
Invalid or incomplete responseI see what you're getting at!

Action:
```
{
  "action": "Fina

{'input': 'Hostel fee is 6000 per month and I stayed for 5 months. Calculate my hostel fee.',
 'output': 'The total hostel fee is indeed 30,000.'}

In [7]:
agent_executor.invoke({"input": "I attended 80 classes out of 100. My marks are 90, 85, 88, 92 and 95. My course fee is 60000 and I paid 45000. Provide: 1. Attendance Status, 2. Grade, 3. Pending Fee."})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "attendance_calculator",
  "action_input": {"total_classes": 100, "attended_classes": 80}
}
```Attendance Percentage: 80.00%
Status: Eligible for ExamAction:
```
{
  "action": "result_calculator",
  "action_input": {"marks_string": "90,85,88,92,95"}
}
```Average Marks: 90.00
Grade: A
Status: PassAction:
```
{
  "action": "fee_balance_calculator",
  "action_input": {"total_course_fee": 60000, "amount_paid": 45000}
}
```Pending Fee Amount: 15000.00Action:
```
{
  "action": "Final Answer",
  "action_input": "Attendance Status: Eligible for Exam, Grade: A, Pass, Pending Fee Amount: 15000.00"
}
```

> Finished chain.


{'input': 'I attended 80 classes out of 100. My marks are 90, 85, 88, 92 and 95. My course fee is 60000 and I paid 45000. Provide: 1. Attendance Status, 2. Grade, 3. Pending Fee.',
 'output': 'Attendance Status: Eligible for Exam, Grade: A, Pass, Pending Fee Amount: 15000.00'}

In [8]:
agent_executor.invoke({"input": "Can you look up the details for student ID STU001?"})



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "student_information_tool",
  "action_input": "STU001"
}
```
ID: STU001
Name: Alice Smith
Major: Computer Science
Year: 2nd YearHere's the response:

Action:
```
{
  "action": "student_information_tool",
  "action_input": "STU001"
}
```
ID: STU001
Name: Alice Smith
Major: Computer Science
Year: 2nd YearI see you're looking for student information! Here's the response:

Action:
```
{
  "action": "student_information_tool",
  "action_input": "STU001"
}
```
ID: STU001
Name: Alice Smith
Major: Computer Science
Year: 2nd YearCould not parse LLM output: Action:
{
  "action": "student_information_tool",
  "action_input": "STU001"
}
Invalid or incomplete responseI see you're looking for student information! Here's the response:

Action:
```
{
  "action": "student_information_tool",
  "action_input": "STU001"
}
```
ID: STU001
Name: Alice Smith
Major: Computer Science
Year: 2nd YearCould not parse LLM output: I see you're looking 

{'input': 'Can you look up the details for student ID STU001?',
 'output': 'Student ID STU001 details retrieved successfully. Name: Alice Smith, Major: Computer Science, Year: 2nd Year.'}